# EDA — Bureau y Bureau Balance (Home Credit Default Risk)

Este notebook analiza las tablas `bureau` y `bureau_balance` del dominio de riesgo de crédito para construir evidencia y variables candidatas para **Track C/modelado**.

Enfoque metodológico:
- SparkSQL para análisis masivo en Glue/Athena.
- PySpark para carga, vistas, persistencia y utilidades.
- `toPandas()` solo en resultados agregados pequeños para visualización.

Objetivo de negocio: entender historial crediticio externo, mora histórica y calidad de datos para mejorar la separación entre clientes `target=0` y `target=1`.

## 1. Configuración inicial

Se definen imports, sesión Spark/Glue, rutas de datos y helpers reutilizables para mantener consistencia con los notebooks de EDA previos.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

BUCKET_NAME  = 'hcdr'
TRUSTED_PATH = 's3://{}/trusted/'.format(BUCKET_NAME)
REFINED_PATH = 's3://{}/refined/eda/'.format(BUCKET_NAME)
TRUSTED_DB   = 'trusted_db'

print('Configuracion inicial OK')

In [ ]:
try:
    from awsglue.context import GlueContext
    from pyspark.context import SparkContext
    sc = SparkContext.getOrCreate()
    glueContext = GlueContext(sc)
    spark = glueContext.spark_session
    print('GlueContext OK')
except Exception:
    spark = (
        SparkSession.builder
        .appName('EDA_BUREAU_BUREAU_BALANCE')
        .config('spark.sql.adaptive.enabled', 'true')
        .config('spark.sql.shuffle.partitions', '120')
        .enableHiveSupport()
        .getOrCreate()
    )
    print('SparkSession fallback OK')

spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

In [ ]:
def sql(query, rows=30, truncate=False):
    result = spark.sql(query)
    result.show(rows, truncate=truncate)
    return result


def save_refined(df, name):
    out = REFINED_PATH + name + '/'
    try:
        df.coalesce(1).write.mode('overwrite').option('header', 'true').csv(out)
        print('Saved on refined/{}'.format(name))
    except Exception as e:
        print('No se pudo guardar {}: {}'.format(name, str(e)))


def plot_bar(pdf, x, y, title, xlabel=None, ylabel=None, rotation=45, color='steelblue'):
    if pdf is None or pdf.empty:
        print('Empty df')
        return
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(pdf[x].astype(str), pdf[y], color=color)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel(xlabel or x)
    ax.set_ylabel(ylabel or y)
    plt.xticks(rotation=rotation, ha='right' if rotation else 'center')
    plt.tight_layout()
    plt.show()


def plot_hbar(pdf, x, y, title, xlabel=None, ylabel=None, color='steelblue'):
    if pdf is None or pdf.empty:
        print('Empty df')
        return
    fig, ax = plt.subplots(figsize=(10, max(4, len(pdf) * 0.4)))
    ax.barh(pdf[y].astype(str), pdf[x], color=color)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel(xlabel or x)
    ax.set_ylabel(ylabel or y)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


def plot_line(pdf, x, y, title, xlabel=None, ylabel=None, color='steelblue'):
    if pdf is None or pdf.empty:
        print('Empty df')
        return
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(pdf[x], pdf[y], marker='o', color=color)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel(xlabel or x)
    ax.set_ylabel(ylabel or y)
    plt.tight_layout()
    plt.show()


def load_trusted(name):
    try:
        df = spark.table('{}.{}'.format(TRUSTED_DB, name))
        print('Glue Catalog OK:', name)
    except Exception:
        path = TRUSTED_PATH + name + '/'
        df = spark.read.parquet(path)
        print('S3 fallback OK:', name)
    df.createOrReplaceTempView(name)
    print('  {:,} rows x {} columns'.format(df.count(), len(df.columns)))
    return df

## 2. Carga de datos

Se cargan las tablas desde Glue Catalog (con fallback a S3 trusted) y se crean vistas temporales para trabajar con SparkSQL.

In [ ]:
bureau = load_trusted('bureau')
bureau_balance = load_trusted('bureau_balance')
application_train = load_trusted('application_train')

app_target = application_train.select('sk_id_curr', 'target')
app_target.createOrReplaceTempView('app_target')

bureau.createOrReplaceTempView('bureau')
bureau_balance.createOrReplaceTempView('bureau_balance')

print('Vistas temporales: bureau, bureau_balance, app_target')

## 3. Descripción general de `bureau`

Esta vista resume tamaño, granularidad y diversidad de estados/tipos de crédito reportados por buró. Es clave para entender cobertura histórica por cliente.

In [ ]:
bureau_overview = sql("""
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT sk_id_curr) AS clientes_unicos,
    COUNT(DISTINCT sk_id_bureau) AS creditos_buro_unicos,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT sk_id_curr), 2) AS creditos_promedio_por_cliente,
    MIN(days_credit) AS days_credit_min,
    MAX(days_credit) AS days_credit_max,
    COUNT(DISTINCT credit_type) AS cantidad_tipos_credito,
    COUNT(DISTINCT credit_active) AS cantidad_estados_credit_active
FROM bureau
""")

save_refined(bureau_overview, 'bureau_overview')

## 4. Distribución de `credit_active`

Se analiza el estado del crédito externo (activo, cerrado, vendido, mala deuda, otros). Esta variable suele correlacionar con exposición y severidad de riesgo.

In [ ]:
bureau_credit_active_distribution = sql("""
WITH base AS (
    SELECT
        CASE
            WHEN lower(credit_active) = 'active' THEN 'Active'
            WHEN lower(credit_active) = 'closed' THEN 'Closed'
            WHEN lower(credit_active) = 'sold' THEN 'Sold'
            WHEN lower(credit_active) IN ('bad debt', 'bad_debt') THEN 'Bad debt'
            ELSE 'Otros'
        END AS credit_active_group
    FROM bureau
)
SELECT
    credit_active_group,
    COUNT(*) AS registros,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_registros
FROM base
GROUP BY credit_active_group
ORDER BY registros DESC
""")

save_refined(bureau_credit_active_distribution, 'bureau_credit_active_distribution')

plot_bar(
    bureau_credit_active_distribution.toPandas(),
    x='credit_active_group',
    y='registros',
    title='Bureau | Distribucion de credit_active',
    xlabel='Estado agrupado',
    ylabel='Registros'
)

## 5. Distribución de `credit_type`

Se identifican los tipos de crédito más frecuentes, su participación y cantidad de clientes únicos afectados. Esto orienta la ingeniería de variables categóricas y de concentración.

In [ ]:
bureau_credit_type_distribution = sql("""
SELECT
    COALESCE(credit_type, 'Unknown') AS credit_type,
    COUNT(*) AS registros,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_registros,
    COUNT(DISTINCT sk_id_curr) AS clientes_unicos
FROM bureau
GROUP BY COALESCE(credit_type, 'Unknown')
ORDER BY registros DESC
""")

save_refined(bureau_credit_type_distribution, 'bureau_credit_type_distribution')

plot_hbar(
    bureau_credit_type_distribution.limit(15).toPandas(),
    x='registros',
    y='credit_type',
    title='Bureau | Top 15 tipos de credito',
    xlabel='Registros',
    ylabel='Tipo de credito'
)

## 6. Análisis de montos

Se resumen métricas robustas de montos (promedio, mediana, p90, máximo y nulos) para evaluar escala, sesgo y completitud de variables financieras en buró.

In [ ]:
bureau_amount_summary = sql("""
WITH base AS (
    SELECT
        amt_credit_sum,
        amt_credit_sum_debt,
        amt_credit_sum_limit,
        amt_credit_sum_overdue
    FROM bureau
),
long_format AS (
    SELECT 'amt_credit_sum' AS variable, amt_credit_sum AS valor FROM base
    UNION ALL
    SELECT 'amt_credit_sum_debt' AS variable, amt_credit_sum_debt AS valor FROM base
    UNION ALL
    SELECT 'amt_credit_sum_limit' AS variable, amt_credit_sum_limit AS valor FROM base
    UNION ALL
    SELECT 'amt_credit_sum_overdue' AS variable, amt_credit_sum_overdue AS valor FROM base
)
SELECT
    variable,
    ROUND(AVG(valor), 2) AS promedio,
    ROUND(percentile_approx(valor, 0.5), 2) AS mediana,
    ROUND(percentile_approx(valor, 0.9), 2) AS p90,
    ROUND(MAX(valor), 2) AS maximo,
    SUM(CASE WHEN valor IS NULL THEN 1 ELSE 0 END) AS nulls,
    ROUND(SUM(CASE WHEN valor IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_nulls
FROM long_format
GROUP BY variable
ORDER BY variable
""")

save_refined(bureau_amount_summary, 'bureau_amount_summary')

In [ ]:
bureau_credit_amount_buckets = sql("""
SELECT
    CASE
        WHEN amt_credit_sum IS NULL THEN 'N/A'
        WHEN amt_credit_sum = 0 THEN '0'
        WHEN amt_credit_sum <= 50000 THEN '1 - 50k'
        WHEN amt_credit_sum <= 100000 THEN '50k - 100k'
        WHEN amt_credit_sum <= 500000 THEN '100k - 500k'
        WHEN amt_credit_sum <= 1000000 THEN '500k - 1M'
        ELSE '> 1M'
    END AS credit_amount_bucket,
    COUNT(*) AS registros,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_registros
FROM bureau
GROUP BY 1
ORDER BY registros DESC
""")

save_refined(bureau_credit_amount_buckets, 'bureau_credit_amount_buckets')

plot_bar(
    bureau_credit_amount_buckets.toPandas(),
    x='credit_amount_bucket',
    y='registros',
    title='Bureau | Buckets de monto de credito',
    xlabel='Bucket',
    ylabel='Registros'
)

## 7. Ratios financieros útiles

Se construyen razones de deuda, mora y límite para normalizar montos por tamaño del crédito y capturar intensidad relativa de riesgo.

In [ ]:
bureau_financial_ratios = sql("""
WITH ratios AS (
    SELECT
        CASE WHEN amt_credit_sum > 0 THEN amt_credit_sum_debt / amt_credit_sum ELSE NULL END AS debt_ratio,
        CASE WHEN amt_credit_sum > 0 THEN amt_credit_sum_overdue / amt_credit_sum ELSE NULL END AS overdue_ratio,
        CASE WHEN amt_credit_sum > 0 THEN amt_credit_sum_limit / amt_credit_sum ELSE NULL END AS limit_ratio
    FROM bureau
),
long_format AS (
    SELECT 'debt_ratio' AS ratio_name, debt_ratio AS ratio_value FROM ratios
    UNION ALL
    SELECT 'overdue_ratio' AS ratio_name, overdue_ratio AS ratio_value FROM ratios
    UNION ALL
    SELECT 'limit_ratio' AS ratio_name, limit_ratio AS ratio_value FROM ratios
)
SELECT
    ratio_name,
    ROUND(AVG(ratio_value), 4) AS avg_ratio,
    ROUND(percentile_approx(ratio_value, 0.5), 4) AS median_ratio,
    ROUND(percentile_approx(ratio_value, 0.9), 4) AS p90_ratio,
    ROUND(MAX(ratio_value), 4) AS max_ratio,
    SUM(CASE WHEN ratio_value IS NULL THEN 1 ELSE 0 END) AS nulls
FROM long_format
GROUP BY ratio_name
ORDER BY ratio_name
""")

save_refined(bureau_financial_ratios, 'bureau_financial_ratios')

## 8. Mora y riesgo

La mora observada en buró es un predictor natural de incumplimiento. Se evalúa severidad por días, montos vencidos y heterogeneidad por estado/tipo de crédito.

In [ ]:
bureau_overdue_distribution = sql("""
WITH base AS (
    SELECT
        COALESCE(credit_day_overdue, 0) AS credit_day_overdue,
        amt_credit_sum_overdue,
        credit_active,
        CASE
            WHEN COALESCE(credit_day_overdue, 0) = 0 THEN 'Sin mora'
            WHEN credit_day_overdue BETWEEN 1 AND 30 THEN '1-30 dias'
            WHEN credit_day_overdue BETWEEN 31 AND 60 THEN '31-60 dias'
            WHEN credit_day_overdue BETWEEN 61 AND 90 THEN '61-90 dias'
            ELSE '90+ dias'
        END AS overdue_bucket
    FROM bureau
),
summary AS (
    SELECT
        overdue_bucket,
        COUNT(*) AS registros,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_registros,
        ROUND(AVG(amt_credit_sum_overdue), 2) AS avg_amt_overdue
    FROM base
    GROUP BY overdue_bucket
)
SELECT * FROM summary
ORDER BY
    CASE overdue_bucket
        WHEN 'Sin mora' THEN 1
        WHEN '1-30 dias' THEN 2
        WHEN '31-60 dias' THEN 3
        WHEN '61-90 dias' THEN 4
        ELSE 5
    END
""")

save_refined(bureau_overdue_distribution, 'bureau_overdue_distribution')

overdue_global = sql("""
SELECT
    COUNT(*) AS total_creditos,
    SUM(CASE WHEN COALESCE(credit_day_overdue, 0) > 0 THEN 1 ELSE 0 END) AS creditos_con_mora,
    ROUND(SUM(CASE WHEN COALESCE(credit_day_overdue, 0) > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_creditos_con_mora,
    ROUND(AVG(COALESCE(credit_day_overdue, 0)), 2) AS avg_days_overdue,
    ROUND(AVG(COALESCE(amt_credit_sum_overdue, 0)), 2) AS avg_amt_credit_sum_overdue
FROM bureau
""")

mora_by_active = sql("""
SELECT
    COALESCE(credit_active, 'Unknown') AS credit_active,
    COUNT(*) AS creditos,
    SUM(CASE WHEN COALESCE(credit_day_overdue, 0) > 0 THEN 1 ELSE 0 END) AS creditos_con_mora,
    ROUND(SUM(CASE WHEN COALESCE(credit_day_overdue, 0) > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_con_mora
FROM bureau
GROUP BY COALESCE(credit_active, 'Unknown')
ORDER BY pct_con_mora DESC
""")

plot_bar(
    bureau_overdue_distribution.toPandas(),
    x='overdue_bucket',
    y='registros',
    title='Bureau | Distribucion de mora por dias',
    xlabel='Bucket de mora',
    ylabel='Registros'
)

In [ ]:
bureau_overdue_by_credit_type = sql("""
SELECT
    COALESCE(credit_type, 'Unknown') AS credit_type,
    COUNT(*) AS creditos,
    SUM(CASE WHEN COALESCE(credit_day_overdue, 0) > 0 THEN 1 ELSE 0 END) AS creditos_con_mora,
    ROUND(SUM(CASE WHEN COALESCE(credit_day_overdue, 0) > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_con_mora,
    ROUND(AVG(COALESCE(credit_day_overdue, 0)), 2) AS avg_days_overdue,
    ROUND(SUM(COALESCE(amt_credit_sum_overdue, 0)), 2) AS total_amt_overdue
FROM bureau
GROUP BY COALESCE(credit_type, 'Unknown')
ORDER BY pct_con_mora DESC, creditos DESC
""")

save_refined(bureau_overdue_by_credit_type, 'bureau_overdue_by_credit_type')

plot_hbar(
    bureau_overdue_by_credit_type.limit(15).toPandas(),
    x='pct_con_mora',
    y='credit_type',
    title='Bureau | Top 15 tipos con mayor % de mora',
    xlabel='% con mora',
    ylabel='Tipo de credito'
)

## 9. Variables temporales

Se derivan variables temporalmente interpretables para capturar antigüedad del crédito, horizonte esperado y recencia de actualización en buró.

In [ ]:
bureau_temporal_summary = sql("""
WITH base AS (
    SELECT
        b.sk_id_curr,
        t.target,
        -b.days_credit / 365.0 AS credit_age_years,
        b.days_credit_enddate / 365.0 AS expected_remaining_years,
        -b.days_enddate_fact / 365.0 AS actual_end_years,
        -b.days_credit_update / 365.0 AS update_age_years
    FROM bureau b
    LEFT JOIN app_target t
        ON b.sk_id_curr = t.sk_id_curr
)
SELECT
    target,
    COUNT(*) AS creditos,
    ROUND(AVG(credit_age_years), 3) AS avg_credit_age_years,
    ROUND(percentile_approx(credit_age_years, 0.5), 3) AS med_credit_age_years,
    ROUND(AVG(expected_remaining_years), 3) AS avg_expected_remaining_years,
    ROUND(AVG(actual_end_years), 3) AS avg_actual_end_years,
    ROUND(AVG(update_age_years), 3) AS avg_update_age_years
FROM base
GROUP BY target
ORDER BY target
""")

save_refined(bureau_temporal_summary, 'bureau_temporal_summary')

## 10. Inconsistencias de negocio

Estas validaciones ayudan a detectar anomalías de calidad de datos. En riesgo de crédito, no siempre se eliminan: también pueden convertirse en *flags* informativos para el modelo.

In [ ]:
bureau_business_inconsistencies = sql("""
WITH total AS (
    SELECT COUNT(*) AS total_reg FROM bureau
),
rules AS (
    SELECT 'Active con days_enddate_fact no nulo' AS regla, *
    FROM bureau
    WHERE lower(credit_active) = 'active' AND days_enddate_fact IS NOT NULL

    UNION ALL

    SELECT 'Closed/Sold con days_enddate_fact nulo' AS regla, *
    FROM bureau
    WHERE lower(credit_active) IN ('closed', 'sold') AND days_enddate_fact IS NULL

    UNION ALL

    SELECT 'amt_credit_sum_debt negativo' AS regla, *
    FROM bureau
    WHERE amt_credit_sum_debt < 0

    UNION ALL

    SELECT 'amt_credit_sum_overdue negativo' AS regla, *
    FROM bureau
    WHERE amt_credit_sum_overdue < 0

    UNION ALL

    SELECT 'amt_credit_sum = 0 y deuda > 0' AS regla, *
    FROM bureau
    WHERE COALESCE(amt_credit_sum, 0) = 0 AND COALESCE(amt_credit_sum_debt, 0) > 0

    UNION ALL

    SELECT 'days_credit positivo' AS regla, *
    FROM bureau
    WHERE days_credit > 0

    UNION ALL

    SELECT 'days_credit_update positivo' AS regla, *
    FROM bureau
    WHERE days_credit_update > 0
)
SELECT
    regla,
    COUNT(*) AS registros_afectados,
    ROUND(COUNT(*) * 100.0 / MAX(total.total_reg), 4) AS pct_sobre_bureau,
    COUNT(DISTINCT sk_id_curr) AS clientes_afectados
FROM rules
CROSS JOIN total
GROUP BY regla
ORDER BY registros_afectados DESC
""")

save_refined(bureau_business_inconsistencies, 'bureau_business_inconsistencies')

**Nota de modelado:** una inconsistencia puede reflejar errores de origen, pero también comportamiento extremo. En Track C conviene evaluar dos enfoques: limpieza (setear nulos/caps) y creación de variables bandera por regla.

## 11. Relación con `target`

Se agregan variables de buró a nivel cliente para contrastar patrones entre `target=0` y `target=1`.

In [ ]:
bureau_agg_target = sql("""
WITH base AS (
    SELECT
        b.sk_id_curr,
        t.target,
        b.credit_active,
        b.credit_type,
        b.amt_credit_sum,
        b.amt_credit_sum_debt,
        b.amt_credit_sum_overdue,
        b.credit_day_overdue,
        b.days_credit,
        b.days_credit_update,
        b.cnt_credit_prolong,
        CASE WHEN b.amt_credit_sum > 0 THEN b.amt_credit_sum_debt / b.amt_credit_sum ELSE NULL END AS debt_ratio
    FROM bureau b
    JOIN app_target t
        ON b.sk_id_curr = t.sk_id_curr
)
SELECT
    sk_id_curr,
    target,
    COUNT(*) AS bureau_credit_count,
    SUM(CASE WHEN lower(credit_active) = 'active' THEN 1 ELSE 0 END) AS bureau_active_credit_count,
    SUM(CASE WHEN lower(credit_active) = 'closed' THEN 1 ELSE 0 END) AS bureau_closed_credit_count,
    COUNT(DISTINCT credit_type) AS bureau_credit_type_count,
    ROUND(SUM(COALESCE(amt_credit_sum, 0)), 2) AS bureau_total_credit_sum,
    ROUND(SUM(COALESCE(amt_credit_sum_debt, 0)), 2) AS bureau_total_debt_sum,
    ROUND(SUM(COALESCE(amt_credit_sum_overdue, 0)), 2) AS bureau_total_overdue_sum,
    MAX(COALESCE(credit_day_overdue, 0)) AS bureau_max_credit_day_overdue,
    ROUND(AVG(-days_credit / 365.0), 4) AS bureau_avg_credit_age_years,
    ROUND(MAX(debt_ratio), 6) AS bureau_debt_ratio_max,
    ROUND(AVG(debt_ratio), 6) AS bureau_debt_ratio_mean,
    SUM(COALESCE(cnt_credit_prolong, 0)) AS bureau_cnt_credit_prolong_sum,
    SUM(CASE WHEN COALESCE(credit_day_overdue, 0) > 0 THEN 1 ELSE 0 END) AS bureau_overdue_credit_count,
    MIN(days_credit_update) AS bureau_recent_update_days
FROM base
GROUP BY sk_id_curr, target
""")

bureau_agg_target.createOrReplaceTempView('bureau_agg_target')

bureau_default_comparison = sql("""
SELECT
    target,
    COUNT(*) AS clientes,
    ROUND(AVG(bureau_credit_count), 3) AS avg_bureau_credit_count,
    percentile_approx(bureau_credit_count, 0.5) AS med_bureau_credit_count,
    percentile_approx(bureau_credit_count, 0.9) AS p90_bureau_credit_count,
    ROUND(AVG(bureau_total_credit_sum), 2) AS avg_bureau_total_credit_sum,
    ROUND(AVG(bureau_total_debt_sum), 2) AS avg_bureau_total_debt_sum,
    ROUND(AVG(bureau_total_overdue_sum), 2) AS avg_bureau_total_overdue_sum,
    ROUND(AVG(bureau_overdue_credit_count), 3) AS avg_bureau_overdue_credit_count,
    ROUND(AVG(bureau_max_credit_day_overdue), 3) AS avg_bureau_max_credit_day_overdue,
    ROUND(AVG(bureau_debt_ratio_mean), 6) AS avg_bureau_debt_ratio_mean,
    ROUND(AVG(bureau_debt_ratio_max), 6) AS avg_bureau_debt_ratio_max,
    ROUND(AVG(bureau_avg_credit_age_years), 4) AS avg_bureau_avg_credit_age_years,
    ROUND(AVG(bureau_recent_update_days), 3) AS avg_bureau_recent_update_days
FROM bureau_agg_target
GROUP BY target
ORDER BY target
""")

save_refined(bureau_default_comparison, 'bureau_default_comparison')

## 12. Análisis de `bureau_balance`

`bureau_balance` contiene snapshots mensuales del estado de cada crédito de buró. Permite caracterizar dinámica de pago/incumplimiento a lo largo del tiempo.

In [ ]:
bureau_balance_overview = sql("""
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT sk_id_bureau) AS creditos_buro_unicos,
    MIN(months_balance) AS months_balance_min,
    MAX(months_balance) AS months_balance_max,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT sk_id_bureau), 2) AS snapshots_promedio_por_credito
FROM bureau_balance
""")

save_refined(bureau_balance_overview, 'bureau_balance_overview')

bureau_balance_status_distribution = sql("""
SELECT
    UPPER(COALESCE(status, 'X')) AS status,
    COUNT(*) AS registros,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_registros,
    ROUND(
        SUM(CASE WHEN UPPER(COALESCE(status, 'X')) IN ('1','2','3','4','5') THEN 1 ELSE 0 END)
        * 100.0 / COUNT(*), 2
    ) AS pct_mora_en_status
FROM bureau_balance
GROUP BY UPPER(COALESCE(status, 'X'))
ORDER BY registros DESC
""")

save_refined(bureau_balance_status_distribution, 'bureau_balance_status_distribution')

bureau_balance_temporal_status = sql("""
SELECT
    months_balance,
    COUNT(*) AS snapshots,
    ROUND(
        SUM(CASE WHEN UPPER(COALESCE(status, 'X')) IN ('1','2','3','4','5') THEN 1 ELSE 0 END)
        * 100.0 / COUNT(*), 2
    ) AS pct_delinquency,
    ROUND(
        SUM(CASE WHEN UPPER(COALESCE(status, 'X')) = 'C' THEN 1 ELSE 0 END)
        * 100.0 / COUNT(*), 2
    ) AS pct_closed,
    ROUND(
        SUM(CASE WHEN UPPER(COALESCE(status, 'X')) = 'X' THEN 1 ELSE 0 END)
        * 100.0 / COUNT(*), 2
    ) AS pct_unknown
FROM bureau_balance
GROUP BY months_balance
ORDER BY months_balance
""")

save_refined(bureau_balance_temporal_status, 'bureau_balance_temporal_status')

plot_bar(
    bureau_balance_status_distribution.toPandas(),
    x='status',
    y='registros',
    title='Bureau Balance | Distribucion de status',
    xlabel='Status',
    ylabel='Registros'
)

plot_line(
    bureau_balance_temporal_status.toPandas(),
    x='months_balance',
    y='pct_delinquency',
    title='Bureau Balance | % mora por mes',
    xlabel='months_balance',
    ylabel='% mora'
)

## 13. Agregación de `bureau_balance` a nivel cliente

Se mapea `bureau_balance` -> `bureau` (`sk_id_bureau`) y luego -> `app_target` (`sk_id_curr`) para construir *features* históricas por cliente.

In [ ]:
bureau_balance_client_features = sql("""
WITH bb_enriched AS (
    SELECT
        b.sk_id_curr,
        t.target,
        bb.sk_id_bureau,
        bb.months_balance,
        UPPER(COALESCE(bb.status, 'X')) AS status
    FROM bureau_balance bb
    JOIN bureau b
        ON bb.sk_id_bureau = b.sk_id_bureau
    JOIN app_target t
        ON b.sk_id_curr = t.sk_id_curr
),
agg AS (
    SELECT
        sk_id_curr,
        target,
        COUNT(*) AS bb_total_snapshots,
        COUNT(DISTINCT months_balance) AS bb_months_observed,
        SUM(CASE WHEN status = '0' THEN 1 ELSE 0 END) AS bb_status_0_count,
        SUM(CASE WHEN status = '1' THEN 1 ELSE 0 END) AS bb_status_1_count,
        SUM(CASE WHEN status = '2' THEN 1 ELSE 0 END) AS bb_status_2_count,
        SUM(CASE WHEN status = '3' THEN 1 ELSE 0 END) AS bb_status_3_count,
        SUM(CASE WHEN status = '4' THEN 1 ELSE 0 END) AS bb_status_4_count,
        SUM(CASE WHEN status = '5' THEN 1 ELSE 0 END) AS bb_status_5_count,
        SUM(CASE WHEN status = 'C' THEN 1 ELSE 0 END) AS bb_status_c_count,
        SUM(CASE WHEN status = 'X' THEN 1 ELSE 0 END) AS bb_status_x_count,
        MAX(
            CASE
                WHEN status IN ('1', '2', '3', '4', '5') THEN CAST(status AS INT)
                WHEN status = '0' THEN 0
                ELSE NULL
            END
        ) AS bb_worst_status_numeric,
        SUM(CASE WHEN status IN ('1','2','3','4','5') THEN 1 ELSE 0 END) AS bb_delinquency_months,
        SUM(CASE WHEN status IN ('3','4','5') THEN 1 ELSE 0 END) AS bb_severe_delinquency_months,
        ROUND(SUM(CASE WHEN status IN ('1','2','3','4','5') THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 4) AS bb_pct_delinquency_months,
        ROUND(SUM(CASE WHEN status = 'C' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 4) AS bb_pct_closed_months,
        ROUND(SUM(CASE WHEN status = 'X' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 4) AS bb_pct_unknown_months
    FROM bb_enriched
    GROUP BY sk_id_curr, target
)
SELECT * FROM agg
""")

bureau_balance_client_features.createOrReplaceTempView('bureau_balance_client_features_v')
save_refined(bureau_balance_client_features, 'bureau_balance_client_features')

## 14. Comparación `bureau_balance` vs `target`

Se contrasta la intensidad de mora histórica y calidad del historial mensual entre clientes con y sin default.

In [ ]:
bureau_balance_default_comparison = sql("""
SELECT
    target,
    COUNT(*) AS clientes,
    ROUND(AVG(bb_total_snapshots), 3) AS avg_bb_total_snapshots,
    ROUND(percentile_approx(bb_total_snapshots, 0.5), 3) AS med_bb_total_snapshots,
    ROUND(AVG(bb_worst_status_numeric), 4) AS avg_bb_worst_status_numeric,
    ROUND(percentile_approx(bb_worst_status_numeric, 0.9), 4) AS p90_bb_worst_status_numeric,
    ROUND(AVG(bb_delinquency_months), 4) AS avg_bb_delinquency_months,
    ROUND(AVG(bb_severe_delinquency_months), 4) AS avg_bb_severe_delinquency_months,
    ROUND(AVG(bb_pct_delinquency_months), 4) AS avg_bb_pct_delinquency_months,
    ROUND(AVG(bb_pct_unknown_months), 4) AS avg_bb_pct_unknown_months,
    ROUND(AVG(bb_pct_closed_months), 4) AS avg_bb_pct_closed_months
FROM bureau_balance_client_features_v
GROUP BY target
ORDER BY target
""")

save_refined(bureau_balance_default_comparison, 'bureau_balance_default_comparison')

## 15. Cruce `bureau` + `bureau_balance`

Se combinan features de ambas fuentes a nivel cliente para explorar si mayor mora histórica se asocia con mayor proporción de `target=1`.

In [ ]:
bureau_combined_features_preview = sql("""
SELECT
    a.sk_id_curr,
    a.target,
    a.bureau_credit_count,
    a.bureau_active_credit_count,
    a.bureau_closed_credit_count,
    a.bureau_credit_type_count,
    a.bureau_total_credit_sum,
    a.bureau_total_debt_sum,
    a.bureau_total_overdue_sum,
    a.bureau_max_credit_day_overdue,
    a.bureau_avg_credit_age_years,
    a.bureau_debt_ratio_mean,
    a.bureau_debt_ratio_max,
    a.bureau_overdue_credit_count,
    a.bureau_recent_update_days,
    b.bb_total_snapshots,
    b.bb_worst_status_numeric,
    b.bb_delinquency_months,
    b.bb_severe_delinquency_months,
    b.bb_pct_delinquency_months,
    b.bb_pct_unknown_months,
    b.bb_pct_closed_months
FROM bureau_agg_target a
LEFT JOIN bureau_balance_client_features_v b
    ON a.sk_id_curr = b.sk_id_curr
   AND a.target = b.target
""")

bureau_combined_features_preview.createOrReplaceTempView('bureau_combined_features_preview')
save_refined(bureau_combined_features_preview, 'bureau_combined_features_preview')

mora_vs_default = sql("""
WITH base AS (
    SELECT
        target,
        CASE
            WHEN COALESCE(bb_worst_status_numeric, 0) = 0 THEN 'Sin mora historica'
            WHEN COALESCE(bb_worst_status_numeric, 0) IN (1, 2) THEN 'Mora leve historica'
            ELSE 'Mora alta historica'
        END AS nivel_mora_historica
    FROM bureau_combined_features_preview
),
agg AS (
    SELECT
        nivel_mora_historica,
        COUNT(*) AS clientes,
        SUM(CASE WHEN target = 1 THEN 1 ELSE 0 END) AS clientes_default
    FROM base
    GROUP BY nivel_mora_historica
)
SELECT
    nivel_mora_historica,
    clientes,
    clientes_default,
    ROUND(clientes_default * 100.0 / clientes, 4) AS default_rate_pct
FROM agg
ORDER BY default_rate_pct DESC
""")

plot_bar(
    mora_vs_default.toPandas(),
    x='nivel_mora_historica',
    y='default_rate_pct',
    title='Default rate por nivel de mora historica',
    xlabel='Nivel mora historica',
    ylabel='Default rate (%)'
)

## 16. Conclusiones finales

### Hallazgos principales del EDA
- La tabla `bureau` aporta señal estructural relevante: cantidad de créditos, estado activo/cerrado, concentración por tipo y severidad de mora.
- Las variables monetarias presentan asimetría fuerte y outliers; por eso mediana y percentiles son más estables que solo el promedio.
- `bureau_balance` agrega valor temporal: meses en mora, peor estado histórico y proporción de estados desconocidos/cerrados.
- Las comparaciones por `target` muestran diferencias útiles en intensidad de deuda y deterioro histórico.

### Columnas que parecen útiles
- Conteos de exposición (`bureau_credit_count`, `bureau_active_credit_count`, `bureau_closed_credit_count`).
- Severidad de mora (`bureau_overdue_credit_count`, `bureau_max_credit_day_overdue`, `bureau_total_overdue_sum`).
- Intensidad financiera (`bureau_total_credit_sum`, `bureau_total_debt_sum`, `bureau_debt_ratio_mean`, `bureau_debt_ratio_max`).
- Historial temporal en buró (`bureau_avg_credit_age_years`, `bureau_recent_update_days`).
- Dinámica mensual de `bureau_balance` (`bb_worst_status_numeric`, `bb_delinquency_months`, `bb_severe_delinquency_months`, `bb_pct_delinquency_months`, `bb_pct_unknown_months`, `bb_pct_closed_months`, `bb_total_snapshots`).

### Columnas candidatas a eliminar o revisar
- Variables con nulos extremos o baja varianza operacional tras agregación.
- Campos con alta ambigüedad semántica si no mejoran separación en validación.

### Columnas que deberían transformarse
- Montos monetarios: aplicar `log1p`, winsorización/capping y escalado robusto.
- Razones (`debt_ratio`, `overdue_ratio`, `limit_ratio`): truncar valores extremos y generar flags de outlier.
- Variables temporales en días: pasar a años/meses para mejor interpretabilidad.

### Inconsistencias a tratar en Track C
- Reglas de calidad detectadas (fechas incompatibles, montos negativos, fechas positivas inesperadas) deben quedar como:
  1) banderas de anomalía, y/o
  2) limpieza controlada (null/cap),
  evaluando impacto en performance de SparkML.

### Features recomendadas mínimas para SparkML
- `bureau_credit_count`
- `bureau_active_credit_count`
- `bureau_closed_credit_count`
- `bureau_total_credit_sum`
- `bureau_total_debt_sum`
- `bureau_debt_ratio_mean`
- `bureau_debt_ratio_max`
- `bureau_overdue_credit_count`
- `bureau_max_credit_day_overdue`
- `bureau_total_overdue_sum`
- `bureau_credit_type_count`
- `bureau_avg_credit_age_years`
- `bureau_recent_update_days`
- `bb_total_snapshots`
- `bb_worst_status_numeric`
- `bb_delinquency_months`
- `bb_severe_delinquency_months`
- `bb_pct_delinquency_months`
- `bb_pct_unknown_months`
- `bb_pct_closed_months`

Estas variables conectan el EDA con el flujo Track C (feature engineering en PySpark + modelo SparkML) y son coherentes con una estrategia académica de credit scoring interpretable.